# Part 3: Multi-Request State

In the previous chapters, we built a basic LLM generation loop ourselves.

We gave the model a prompt, ran the prompt through the model once during **prefill**, stored the KV cache, and then generated new tokens one at a time during **decode**.

That worked, but there was one big limitation:

**our engine could only keep track of one request.**

For example, imagine two users send requests at almost the same time:

```text
Request A: "Explain CUDA graphs"

Request B: "Write a poem about GPUs"
```

Each request has its own data.

It has its own:

* input tokens
* generated tokens
* KV cache
* generation progress
* finished/not-finished state

So we cannot keep using one global `input_ids`, one `past_key_values`, and one generated-token list.

We need some way to say:

```text
this KV cache belongs to Request A

this generated token belongs to Request B

Request A is still generating

Request B has finished
```

This is what **request state** gives us.

Instead of the generation loop itself holding all the information, we will create an object representing one active request.

Conceptually:

```text
Request
│
├── request ID
├── input tokens
├── generated tokens
├── KV cache
└── finished?
```

Then we can have multiple independent requests alive at the same time:

```text
Request A ──> its own state
Request B ──> its own state
Request C ──> its own state
```

We are **not doing batching yet**.

That distinction is important.

Multi-request state only means:

> our engine can keep track of multiple generations independently.

We could still run them inefficiently like this:

```text
decode A
decode B
decode C

decode A
decode B
decode C
```

Later, we will build a scheduler that decides which requests should run, and after that we will combine requests into batches so the GPU can process multiple requests together.

For now, the goal is much smaller:

> take the information that belongs to one generation and store it inside a request object.

This is the first step from a simple generation script toward an actual LLM serving engine.

Since this notebook should also work on its own, we will first load PyTorch, the tokenizer, and the model again before building the request-state system.


In [1]:
# Install the Hugging Face libraries we need.
!pip install -q transformers accelerate

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache


# We will keep using the same small Qwen model from the previous chapters.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


# Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# Load the language model onto the GPU.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda"
)


# We are only doing inference, so the model should be in evaluation mode.
model.eval()


print("Model loaded!")
print("Device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!
Device: cuda:0


In [2]:
class RequestState:
  def __init__(self,request_id,prompt):
    self.request_id=request_id
    self.prompt=prompt
    self.input_ids=tokenizer(prompt,return_tensors='pt').input_ids.to(model.device)
    self.generated_tokens=[]
    self.cache=DynamicCache()
    self.finished= False

In [3]:
request_a = RequestState(request_id=0,prompt="The capital of France is")
request_b = RequestState(request_id=1,prompt="The largest planet in our solar system is")
print("Request A ID:", request_a.request_id)
print("Request A input shape:", request_a.input_ids.shape)
print("Request A finished:", request_a.finished)

print()

print("Request B ID:", request_b.request_id)
print("Request B input shape:", request_b.input_ids.shape)
print("Request B finished:", request_b.finished)

Request A ID: 0
Request A input shape: torch.Size([1, 5])
Request A finished: False

Request B ID: 1
Request B input shape: torch.Size([1, 8])
Request B finished: False


## Building Request State

Our previous generation loop only handled one request, so variables such as `input_ids`, the KV cache, and generated tokens could exist globally.

That stops working once multiple requests are active at the same time.

For example:

```text
Request A: "The capital of France is"
Request B: "The largest planet in our solar system is"
```

Both requests need their own:

```text
input tokens
generated tokens
KV cache
completion status
```

So instead of keeping this information inside the generation loop, we will store it inside an object representing one request.

### RequestState

```python
class RequestState:
    def __init__(self, request_id, prompt):
        self.request_id = request_id
        self.prompt = prompt

        self.input_ids = tokenizer(
            prompt,
            return_tensors="pt"
        ).input_ids.to(model.device)

        self.generated_tokens = []

        self.cache = DynamicCache()

        self.finished = False
```

Each `RequestState` object now contains everything that belongs to one generation.

Conceptually:

```text
RequestState
│
├── request_id
├── prompt
├── input_ids
├── generated_tokens
├── KV cache
└── finished
```

### `request_id`

```python
self.request_id = request_id
```

The request ID lets the engine distinguish one request from another.

For example:

```text
Request A → ID 0
Request B → ID 1
```

Later, when the scheduler manages many requests, we need a reliable way to identify which state belongs to which request.

### Tokenizing the prompt

```python
self.input_ids = tokenizer(
    prompt,
    return_tensors="pt"
).input_ids.to(model.device)
```

The tokenizer converts the text prompt into token IDs.

For example:

```text
"The capital of France is"

        ↓ tokenizer

tensor([[..., ..., ..., ..., ...]])
```

The resulting tensor has shape:

```text
[batch_size, sequence_length]
```

Since each request currently contains one sequence:

```text
batch_size = 1
```

A request with five prompt tokens therefore has:

```text
torch.Size([1, 5])
```

The tensor is also moved to `model.device`, because the model and its input tensors need to be on the same device.

### Generated tokens

```python
self.generated_tokens = []
```

When a request is first created, the model has not generated anything yet.

So the list starts empty.

During decoding, new token IDs will be added here:

```text
[]

↓ decode

[token_1]

↓ decode

[token_1, token_2]

↓ decode

[token_1, token_2, token_3]
```

For now, we keep the original prompt tokens and newly generated tokens separate:

```text
input_ids
    → prompt tokens

generated_tokens
    → tokens produced during decoding
```

### Per-request KV cache

```python
self.cache = DynamicCache()
```

This is one of the most important changes.

Previously, our single-request generation loop used one KV cache.

Now every request owns its own cache.

```text
Request A
└── KV cache A

Request B
└── KV cache B
```

Suppose the prompts are:

```text
A → "The capital of France is"

B → "The largest planet in our solar system is"
```

After prefill, cache A will contain the key/value tensors produced from Request A's prompt, while cache B will contain those produced from Request B's prompt.

The caches must remain separate because the KV cache represents the context that a particular request has already processed.

Later, when we build a proper KV-cache manager, this same idea will remain, although the physical storage will become more sophisticated.

### Finished state

```python
self.finished = False
```

A newly created request has not finished generation, so it starts as:

```text
finished = False
```

Later, the request may reach an EOS token or a generation limit:

```python
request.finished = True
```

The scheduler will use this information to avoid scheduling requests that have already completed.

---

## Creating Multiple Requests

We can now create two independent request states:

```python
request_a = RequestState(
    request_id=0,
    prompt="The capital of France is"
)

request_b = RequestState(
    request_id=1,
    prompt="The largest planet in our solar system is"
)
```

Inspecting them:

```python
print(
    "Request A ID:", request_a.request_id,
    "Request A input shape:", request_a.input_ids.shape,
    "Request A finished:", request_a.finished
)

print()

print(
    "Request B ID:", request_b.request_id,
    "Request B input shape:", request_b.input_ids.shape,
    "Request B finished:", request_b.finished
)
```

Output:

```text
Request A ID: 0
Request A input shape: torch.Size([1, 5])
Request A finished: False

Request B ID: 1
Request B input shape: torch.Size([1, 8])
Request B finished: False
```

The sequence lengths are different:

```text
Request A → 5 tokens
Request B → 8 tokens
```

That is completely fine.

We are not batching these requests together yet.

At this stage, we have simply moved from:

```text
one generation
one state
```

to:

```text
multiple generations
each with independent state
```

This separation is the foundation we need before introducing scheduling and continuous batching.


In [7]:
@torch.no_grad()
def prefill(request):
  outputs=model(input_ids=request.input_ids,past_key_values=request.cache,use_cache=True)
  logits=outputs.logits
  next_token_logits=logits[:,-1,:]
  next_token_id=torch.argmax(next_token_logits,dim=-1)
  request.generated_tokens.append(next_token_id.item())
  return next_token_id

In [10]:
next_token_a = prefill(request_a)

print("Next token ID:", next_token_a)
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())

Next token ID: tensor([12095], device='cuda:0')
Generated tokens: [12095]
Cache length: 5


## Prefill a Request

We now have multiple request objects, but their KV caches are still empty.

The next step is to run the prompt of one request through the model.

This is the **prefill** stage.

During prefill:

```text
prompt tokens
    ↓
model forward pass
    ↓
KV cache is created
    ↓
logits for every prompt position
    ↓
use the final position to choose the first generated token
```

We will make prefill operate on a `RequestState` object instead of passing the prompt, cache, and generated tokens around separately.

```python
@torch.no_grad()
def prefill(request):
    outputs = model(
        input_ids=request.input_ids,
        past_key_values=request.cache,
        use_cache=True
    )

    logits = outputs.logits

    next_token_logits = logits[:, -1, :]

    next_token_id = torch.argmax(
        next_token_logits,
        dim=-1
    )

    request.generated_tokens.append(
        next_token_id.item()
    )

    return next_token_id
```

### Why `@torch.no_grad()`?

```python
@torch.no_grad()
```

This is a Python decorator.

It tells PyTorch not to track gradients while this function runs.

Gradients are required during training because PyTorch needs them for backpropagation.

Here we are only running inference, so keeping the gradient computation graph would waste memory.

The same idea could also be written as:

```python
with torch.no_grad():
    ...
```

Using the decorator simply applies it to the entire function.

---

### Running the prompt through the model

```python
outputs = model(
    input_ids=request.input_ids,
    past_key_values=request.cache,
    use_cache=True
)
```

Suppose the request has:

```text
request.input_ids.shape = [1, 5]
```

This means:

```text
1 sequence
5 prompt tokens
```

All five prompt tokens are processed together during prefill.

We also pass:

```python
past_key_values=request.cache
```

Each request owns its own `DynamicCache`, so the K/V tensors produced from this prompt are stored in that request's cache.

After this forward pass:

```text
request.cache length = 5
```

The cache now represents the five prompt tokens that have already gone through the model.

---

### Logits shape

```python
logits = outputs.logits
```

For an input shape of:

```text
[1, 5]
```

the logits have shape:

```text
[1, 5, vocab_size]
```

The dimensions mean:

```text
1
→ number of sequences

5
→ number of token positions

vocab_size
→ score for every possible next token
```

The model produces vocabulary scores at every position, but for generation we only need the prediction after the final prompt token.

So we use:

```python
next_token_logits = logits[:, -1, :]
```

Before:

```text
[1, 5, vocab_size]
```

After:

```text
[1, vocab_size]
```

Here:

```text
:
→ every item in the batch

-1
→ final sequence position

:
→ every vocabulary score
```

---

### Choosing the first generated token

```python
next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)
```

The current tensor shape is:

```text
[1, vocab_size]
```

`dim=-1` means perform `argmax` across the final dimension, which is the vocabulary dimension.

So:

```text
[1, vocab_size]
        ↓
      argmax
        ↓
[1]
```

The result contains the ID of the highest-scoring next token.

For now we are still using greedy decoding, so we always choose the token with the largest logit.

---

### Saving the generated token

```python
request.generated_tokens.append(
    next_token_id.item()
)
```

`next_token_id` is a PyTorch tensor such as:

```text
tensor([12095], device='cuda:0')
```

`.item()` extracts the Python integer:

```text
12095
```

We then store that token inside the current request.

This means each request maintains its own generated sequence:

```text
Request A
└── generated_tokens

Request B
└── generated_tokens
```

---

## Running Prefill

We can now run prefill on Request A:

```python
next_token_a = prefill(request_a)

print("Next token ID:", next_token_a)
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())
```

Example output:

```text
Next token ID: tensor([12095], device='cuda:0')
Generated tokens: [12095]
Cache length: 5
```

There is an important detail here.

We have:

```text
prompt tokens      = 5
generated tokens   = 1
cache length       = 5
```

Why is the cache length not 6?

Because the newly selected token has not gone through the model yet.

Prefill works like this:

```text
5 prompt tokens
      ↓
model
      ↓
cache contains those 5 tokens
      ↓
logits predict token #6
      ↓
choose token #6
```

The model has **predicted** token #6, but we have not fed token #6 back into the model.

That happens during the first decode step.

So the KV cache should be understood as:

> the tokens that have already been processed by the model.

At this point:

```text
processed by model:
prompt tokens 1–5

predicted but not yet processed:
token 6
```

The next stage will feed token #6 into the model, extend the cache from 5 to 6, and use that forward pass to predict token #7.


In [12]:
@torch.no_grad()
def decode_one_token(request):
    # Take the most recently generated token.
    last_token_id = request.generated_tokens[-1]

    # Convert it back into a tensor.
    # Shape: [1, 1]
    input_token = torch.tensor([[last_token_id]],device=model.device)

    outputs = model(input_ids=input_token,past_key_values=request.cache,use_cache=True)
    logits = outputs.logits
    next_token_logits = logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits,dim=-1)
    request.generated_tokens.append(next_token_id.item())

    return next_token_id

In [13]:
next_token_a = decode_one_token(request_a)

print("Next token:", next_token_a)
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())

Next token: tensor([13], device='cuda:0')
Generated tokens: [12095, 13]
Cache length: 6


## Decode One Token

After prefill, Request A currently looks like this:

```text
prompt tokens      = 5
generated tokens   = 1
KV cache length    = 5
```

The first generated token has been **predicted**, but it has not gone through the model yet.

The decode stage takes that newest token, runs it through the model using the existing KV cache, extends the cache by one position, and predicts the next token.

```python
@torch.no_grad()
def decode_one_token(request):
    # Take the most recently generated token.
    last_token_id = request.generated_tokens[-1]

    # Convert the Python integer back into a tensor.
    # Shape: [1, 1]
    input_token = torch.tensor(
        [[last_token_id]],
        device=model.device
    )

    outputs = model(
        input_ids=input_token,
        past_key_values=request.cache,
        use_cache=True
    )

    # Shape: [1, 1, vocab_size]
    logits = outputs.logits

    # Take the logits from the final sequence position.
    # Shape: [1, vocab_size]
    next_token_logits = logits[:, -1, :]

    # Greedy decoding: choose the highest-scoring token.
    # Shape: [1]
    next_token_id = torch.argmax(
        next_token_logits,
        dim=-1
    )

    # Save the newly predicted token in this request.
    request.generated_tokens.append(
        next_token_id.item()
    )

    return next_token_id
```

### Taking the latest generated token

```python
last_token_id = request.generated_tokens[-1]
```

After prefill, our list contains one token:

```text
generated_tokens = [12095]
```

`[-1]` means the last item in the list, so:

```text
last_token_id = 12095
```

This is the token we need to feed into the model next.

We do **not** send the entire prompt again.

The prompt has already been processed and its information is stored in the KV cache.

---

### Creating the decode input

The token stored in `generated_tokens` is a Python integer.

The model expects its input as a PyTorch tensor with shape:

```text
[batch_size, sequence_length]
```

So we convert it:

```python
input_token = torch.tensor(
    [[last_token_id]],
    device=model.device
)
```

For example:

```text
12095

↓
tensor([[12095]])
```

The shape is:

```text
[1, 1]
```

because we currently have:

```text
1 request
1 new token
```

This is different from prefill.

During prefill we may have had:

```text
input shape = [1, 5]
```

because the complete five-token prompt was processed at once.

During decode:

```text
input shape = [1, 1]
```

because only the newest token needs to be processed.

---

## Using the Existing KV Cache

```python
outputs = model(
    input_ids=input_token,
    past_key_values=request.cache,
    use_cache=True
)
```

Before this call, the state is:

```text
KV cache:
prompt tokens 1–5

new input:
token 6
```

The model can therefore use the cached K/V tensors for the previous five tokens instead of recomputing them.

Conceptually:

```text
cached tokens 1–5
        +
new token 6
        ↓
      model
        ↓
cache now represents tokens 1–6
        ↓
predict token 7
```

This is the main reason the KV cache matters during autoregressive decoding.

Without it, every generation step would have to process the complete sequence again:

```text
step 1 → process tokens 1–5
step 2 → process tokens 1–6
step 3 → process tokens 1–7
...
```

With the KV cache, the old tokens have already been processed:

```text
prefill → process tokens 1–5

decode → process only token 6

decode → process only token 7

decode → process only token 8
```

---

## Decode Logits Shape

After the model call:

```python
logits = outputs.logits
```

the logits have shape:

```text
[1, 1, vocab_size]
```

Compare this with prefill:

```text
Prefill input:
[1, 5]

Prefill logits:
[1, 5, vocab_size]
```

During decode:

```text
Decode input:
[1, 1]

Decode logits:
[1, 1, vocab_size]
```

The sequence dimension is now `1` because only one new token went through the model.

We still write:

```python
next_token_logits = logits[:, -1, :]
```

which gives:

```text
[1, vocab_size]
```

Even though there is currently only one sequence position, using `-1` keeps the logic consistent: we always select the logits from the final processed position.

---

## Predicting the Next Token

```python
next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)
```

The shape before `argmax` is:

```text
[1, vocab_size]
```

`dim=-1` means:

> search across the final dimension.

The final dimension is the vocabulary dimension.

So we go from:

```text
[1, vocab_size]

      ↓ argmax

[1]
```

and obtain the ID of the highest-scoring next token.

We then save it:

```python
request.generated_tokens.append(
    next_token_id.item()
)
```

---

## Running One Decode Step

```python
next_token_a = decode_one_token(request_a)

print("Next token:", next_token_a)
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())
```

Output:

```text
Next token: tensor([13], device='cuda:0')
Generated tokens: [12095, 13]
Cache length: 6
```

The state has changed from:

```text
Before decode:

cache length      = 5
generated tokens  = [12095]
```

to:

```text
After decode:

cache length      = 6
generated tokens  = [12095, 13]
```

Why did the cache increase from `5` to `6`?

Because token `12095`, which was predicted during prefill, has now gone through the model.

So its K/V tensors have been added to the cache.

The newly predicted token `13` has **not** gone through the model yet.

The state is therefore:

```text
processed by model:
prompt tokens 1–5 + generated token 6

KV cache length:
6

predicted but not yet processed:
generated token 7
```

On the next call to `decode_one_token()`, token 7 will be fed into the model, the cache will grow from `6` to `7`, and token 8 will be predicted.

This one-token-at-a-time state transition is the core of autoregressive decoding with a KV cache.


In [14]:
#decoding for n tokens

def decode_n_tokens(request, n):
    for _ in range(n):
        decode_one_token(request)

In [15]:
decode_n_tokens(request_a, 5)

print("Generated token IDs:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())

Generated token IDs: [12095, 13, 1084, 374, 279, 7772, 3283]
Cache length: 11


In [16]:
#Then let's actually see what the model generated:

text = tokenizer.decode(request_a.generated_tokens,skip_special_tokens=True)

print(text)

 Paris. It is the largest city


## Decode Multiple Tokens

`decode_one_token()` generates one new token at a time. To continue generation, we can simply call it repeatedly.

```python
def decode_n_tokens(request, n):
    for _ in range(n):
        decode_one_token(request)
```

Here `_` means we do not care about the loop counter. We only want to repeat the decode step `n` times.

Now generate five more tokens:

```python
decode_n_tokens(request_a, 5)

print("Generated token IDs:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())
```

Output:

```text
Generated token IDs: [12095, 13, 1084, 374, 279, 7772, 3283]
Cache length: 11
```

Each decode step adds:

```text
1 token to generated_tokens
1 processed token to the KV cache
```

We can convert the generated token IDs back into readable text:

```python
text = tokenizer.decode(
    request_a.generated_tokens,
    skip_special_tokens=True
)

print(text)
```

Output:

```text
 Paris. It is the largest city
```

So our request can now keep its own KV cache and continue decoding for multiple steps.


In [17]:
#repeating same for b

next_token_b = prefill(request_b)

print("Request A")
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())

print()

print("Request B")
print("Generated tokens:", request_b.generated_tokens)
print("Cache length:", request_b.cache.get_seq_length())

Request A
Generated tokens: [12095, 13, 1084, 374, 279, 7772, 3283]
Cache length: 11

Request B
Generated tokens: [1304]
Cache length: 8


In [18]:
#alternating between them

decode_one_token(request_a)
decode_one_token(request_b)

decode_one_token(request_a)
decode_one_token(request_b)

tensor([32], device='cuda:0')

In [19]:
print("A:", tokenizer.decode(request_a.generated_tokens))
print("A cache:", request_a.cache.get_seq_length())

print()

print("B:", tokenizer.decode(request_b.generated_tokens))
print("B cache:", request_b.cache.get_seq_length())

A:  Paris. It is the largest city in Europe
A cache: 13

B:  ____
A
B cache: 10


## Multiple Independent Requests

So far, only Request A has been generating tokens. Request B still has an empty KV cache.

We can prefill Request B independently:

```python id="sn2zi3"
next_token_b = prefill(request_b)

print("Request A")
print("Generated tokens:", request_a.generated_tokens)
print("Cache length:", request_a.cache.get_seq_length())

print()

print("Request B")
print("Generated tokens:", request_b.generated_tokens)
print("Cache length:", request_b.cache.get_seq_length())
```

Output:

```text id="g8fk88"
Request A
Generated tokens: [12095, 13, 1084, 374, 279, 7772, 3283]
Cache length: 11

Request B
Generated tokens: [1304]
Cache length: 8
```

The important point is that pre-filling Request B does not change Request A.

Each request owns its own state:

```text id="95srtw"
Request A
├── generated tokens
└── KV cache

Request B
├── generated tokens
└── KV cache
```

We can now alternate decode steps between the two requests:

```python id="r8stj9"
decode_one_token(request_a)
decode_one_token(request_b)

decode_one_token(request_a)
decode_one_token(request_b)
```

Then inspect both:

```python id="1htvp3"
print("A:", tokenizer.decode(request_a.generated_tokens))
print("A cache:", request_a.cache.get_seq_length())

print()

print("B:", tokenizer.decode(request_b.generated_tokens))
print("B cache:", request_b.cache.get_seq_length())
```

Example output:

```text id="z8cafc"
A:  Paris. It is the largest city in Europe
A cache: 13

B:  ____
A
B cache: 10
```

Each request received two decode steps, so both KV caches grew by two:

```text id="9l18ud"
Request A: 11 → 13
Request B:  8 → 10
```

This is our first real multi-request system.

The requests are still executed one at a time, but their generation state is completely independent.

For now, we are manually deciding the execution order:

```text id="mgudt4"
A
B
A
B
```
In the next part, we will replace this manual ordering with a small scheduler

Note: The wierd output of B isn't an engine bug. We're using an Instruct model with raw text prompts instead of its chat template, plus greedy decoding, so completions can sometimes be weird. For learning the serving mechanics, that's fine.

Also this part may have felt shorter because the interesting complexity people associate with “serving multiple users” actually belongs to the next few parts, not to request state itself.